In [1]:
# Configuración de entorno y logging
import os
import sys
import logging
import pandas as pd

# Configurar logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# Agregar src/ al path si no está
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [2]:
# Importar funciones y cargar archivo csv
from src.data.load_data import cargar_datos_tipo_data

columnas_adult = [
    "age", "workclass", "fnlwgt", "education", "education-num",
    "marital-status", "occupation", "relationship", "race", "sex",
    "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"
]

# Ruta del archivo CSV
ruta_csv = "../data/raw/adult.data"

# Sw carga la data
df = cargar_datos_tipo_data(ruta_csv, columnas_adult)

2025-07-10 18:29:47,218 - INFO - Datos cargados correctamente desde: ../data/raw/adult.data


In [3]:
# Análisis exploratorio de la data
from src.data.exploracion_inicial import diagnostico_inicial_numericas, diagnostico_inicial_categoricas

# Análisis variables numéricas
diagnostico_inicial_numericas(df, save_dir="../outputs/01_diagnostico")

# Análisis variables categóricas
diagnostico_inicial_categoricas(df,
    save_dir="../outputs/01_diagnostico",
    top_n=5
)

2025-07-10 18:29:51,843 - INFO - Resumen general del dataset:
2025-07-10 18:29:51,844 - INFO - Filas: 32561, Columnas: 15
2025-07-10 18:29:51,847 - INFO - Columnas:
age                int64
workclass         object
fnlwgt             int64
education         object
education-num      int64
marital-status    object
occupation        object
relationship      object
race              object
sex               object
capital-gain       int64
capital-loss       int64
hours-per-week     int64
native-country    object
income            object
dtype: object
2025-07-10 18:29:51,881 - INFO - Resumen estadístico:
            age      fnlwgt  education-num  capital-gain  capital-loss  \
count  32561.00    32561.00       32561.00      32561.00      32561.00   
mean      38.58   189778.37          10.08       1077.65         87.30   
std       13.64   105549.98           2.57       7385.29        402.96   
min       17.00    12285.00           1.00          0.00          0.00   
25%       28.00   1178

,variable,n_clases,categorias
0,workclass,9,"State-gov, Self-emp-not-inc, Private, Federal-..."
1,education,16,"Bachelors, HS-grad, 11th, Masters, 9th, Some-c..."
2,marital-status,7,"Never-married, Married-civ-spouse, Divorced, M..."
3,occupation,15,"Adm-clerical, Exec-managerial, Handlers-cleane..."
4,relationship,6,"Not-in-family, Husband, Wife, Own-child, Unmar..."
5,race,5,"White, Black, Asian-Pac-Islander, Amer-Indian-..."
6,sex,2,"Male, Female"
7,native-country,42,"United-States, Cuba, Jamaica, India, ?, Mexico..."
8,income,2,"<=50K, >50K"


Se detectan valores nulos con formato de "?" en las variables "workclass", "occupation" y "native_country"

In [4]:
# Calcular proporción de valores faltantes
variables = ["workclass", "occupation", "native-country"]

for var in variables:
    total = df[var].shape[0]
    faltantes = (df[var] == "?").sum()
    proporcion = faltantes / total
    logging.info(f"{var}: {faltantes} valores '?' ({proporcion:.2%})")

2025-07-10 18:30:19,876 - INFO - workclass: 1836 valores '?' (5.64%)
2025-07-10 18:30:19,880 - INFO - occupation: 1843 valores '?' (5.66%)
2025-07-10 18:30:19,884 - INFO - native-country: 583 valores '?' (1.79%)


Se ha observado que hay variables como "capital-gain" y "capital-loss" que tienen un exceso de valores 0. Se calcula la proporción de 0s en las variables. Esto puede generar distribuciones con una asimetría muy marcada.

In [5]:
from src.data.exploracion_inicial import calcular_proporcion_ceros

resumen_ceros = calcular_proporcion_ceros(
    df,
    save_path="../outputs/01_diagnostico/resumen_ceros.html"
)

2025-07-10 18:30:27,416 - INFO - No se especificaron columnas. Se analizarán todas las numéricas: ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
2025-07-10 18:30:27,424 - INFO - Resumen de ceros guardado en HTML: ../outputs/01_diagnostico/resumen_ceros.html


Las variables 'capital-gain' y 'capital-loss' tienen una gran proporción de 0s para ser variables numéricas continuas, con un 92% y 95% respectivamente. En tal sentido, están altamente sesgadas, por lo que se recomienda convertirlas a categóricas binarias.